## Write a solution to report all the duplicate emails. Note that it's guaranteed that the email field is not NULL.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Person table:
+----+---------+
| id | email   |
+----+---------+
| 1  | a@b.com |
| 2  | c@d.com |
| 3  | a@b.com |
+----+---------+

## import spark Session 

In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Duplicate Email").getOrCreate()

In [12]:
import pandas as pd
data = [[1, 'a@b.com'], [2, 'c@d.com'], [3, 'a@b.com']]
person = pd.DataFrame(data, columns=['id', 'email']).astype({'id':'Int64', 'email':'object'})

In [13]:
df=spark.createDataFrame(person)
df.show()

+---+-------+
| id|  email|
+---+-------+
|  1|a@b.com|
|  2|c@d.com|
|  3|a@b.com|
+---+-------+



In [15]:
df.select("email").groupBy("email").count().filter("count(*)>1").show()


+-------+-----+
|  email|count|
+-------+-----+
|a@b.com|    2|
+-------+-----+



## create a temp table

In [16]:
df.createOrReplaceTempView("Person")

In [19]:
df2=spark.sql("""
          with per_cte as(
          select
                email,
                dense_rank() over(partition by email order by id) as rnk
        from Person
          )
          select
                email 
            from per_cte
          where rnk >1
          """)

In [20]:
df2.show()

+-------+
|  email|
+-------+
|a@b.com|
+-------+

